In [123]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="llama3:latest",
)
llm.invoke("Tell me a joke")

"Here's one:\n\nWhy couldn't the bicycle stand up by itself?\n\n(Wait for it...)\n\nBecause it was two-tired!\n\nHope that made you smile! Do you want to hear another one?"

In [124]:
query = "Tell me a joke"

for chunks in llm.stream(query):
    print(chunks)

Why
 couldn
't
 the
 bicycle
 stand
 up
 by
 itself
?


Because
 it
 was
 two
-t
ired
!


(S
orry
,
 I
 know
 it
's
 a
 bit
 of
 a
 "
dad
"
 joke
,
 but
 I
 hope
 it
 brought
 a
 smile
 to
 your
 face
!)



In [125]:
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(
    model="llama3",
)
prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")

chain = prompt | llm | StrOutputParser()

print(
    chain.invoke(
        {
            "topic": "Space travel",
        }
    )
)

Why did the astronaut break up with his girlfriend before going to Mars?

Because he needed space! (get it?)


In [126]:
topic = {"topic": "Space travel"}

for chunks in chain.stream(topic):
    print(chunks)

Why
 did
 the
 astronaut
 break
 up
 with
 his
 girlfriend
 before
 going
 to
 Mars
?


Because
 he
 needed
 space
!



In [127]:
topic = {"topic": "Space travel"}

async for chunks in chain.astream(topic):
    print(chunks)

Why
 did
 the
 astronaut
 break
 up
 with
 his
 girlfriend
 before
 going
 to
 Mars
?


Because
 he
 needed
 space
!
 (
get
 it
?)



In [128]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(
    model="llama3",
    format="json",
    temperature=0,
)

In [129]:
from langchain_core.messages import HumanMessage

messages = [
    HumanMessage(
        content="What color is the sky at different times of the day? Respond using JSON"
    )
]

chat_model_response = llm.invoke(messages)
print(chat_model_response)

content='{ "times_of_day": [\n  {"time": "dawn", "color": "pinkish-orange"},\n  {"time": "morning", "color": "light blue"},\n  {"time": "midday", "color": "bright blue"},\n  {"time": "afternoon", "color": "hazy blue"},\n  {"time": "sunset", "color": "orange-red"}\n] }\n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n \n\n' id='run-901f9aaf-289d-4ed9-a081-95888f807ba1-0'


In [130]:
import json

from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

json_schema = {
    "title": "Person",
    "description": "Identifying information about a person.",
    "type": "object",
    "properties": {
        "name": {"title": "Name", "description": "The person's name", "type": "string"},
        "age": {"title": "Age", "description": "The person's age", "type": "integer"},
        "fav_food": {
            "title": "Fav Food",
            "description": "The person's favorite food",
            "type": "string",
        },
    },
    "required": ["name", "age"],
}

llm = ChatOllama(model="llama3")

messages = [
    HumanMessage(
        content="Please tell me about a person using the following JSON schema:"
    ),
    HumanMessage(content="{dumps}"),
    HumanMessage(
        content="Now, considering the schema, tell me about a person named John who is 35 years old and loves pizza."
    ),
]

prompt = ChatPromptTemplate.from_messages(messages)
dumps = json.dumps(
    json_schema,
    indent=2,
)

chain = prompt | llm | StrOutputParser()

print(
    chain.invoke(
        {
            "dumps": dumps,
        }
    )
)

I'm happy to help!

However, I don't see any JSON schema provided. Could you please share the schema with me?

Assuming it's a basic person schema, here's what I can do:

Based on the assumption that the JSON schema looks something like this:
```
{
  "name": {"type": "string"},
  "age": {"type": "integer"},
  "hobbies": {"type": "array", "items": {"type": "string"}}
}
```
Let me tell you about John:

John is a person who is **35 years old**. His name is **John**, and his age is stored in the `age` field.

As for his hobbies, John **loves pizza**, which means that `"pizza"` will be an item in his `hobbies` array.

So, the JSON representation of John would look like this:
```json
{
  "name": "John",
  "age": 35,
  "hobbies": ["pizza"]
}
```
Please note that this is just an assumption based on a basic person schema. If your actual schema is different, please feel free to share it, and I'll be happy to help with more accuracy!


In [131]:
from langchain_experimental.llms.ollama_functions import OllamaFunctions

llm = OllamaFunctions(
    model="llama3",
    format="json",
)

In [132]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content="J'adore le programmation.", id='run-8490a56d-d8ef-4934-924b-64af9b3a9fa9-0')

In [133]:
ai_msg.content

"J'adore le programmation."

In [134]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}.",
        ),
        ("human", "{input}"),
    ]
)

chain = prompt | llm
chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

AIMessage(content='Ich liebe auch Programmieren!', id='run-458605f9-a904-4639-88ba-65fdcf9ac696-0')

In [135]:
from langchain_core.pydantic_v1 import BaseModel, Field


class GetWeather(BaseModel):
    """Get the current weather in a given location"""

    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")


llm_with_tools = llm.bind_tools([GetWeather])

In [136]:
ai_msg = llm_with_tools.invoke(
    "what is the weather like in San Francisco",
)
ai_msg

AIMessage(content='', id='run-7d412265-5a3e-4ef9-9b32-e7fcacf477f1-0', tool_calls=[{'name': 'GetWeather', 'args': {'location': 'San Francisco, CA'}, 'id': 'call_a8669dd5697f488a8c5760e1d37b4528', 'type': 'tool_call'}])

In [137]:
ai_msg.tool_calls

[{'name': 'GetWeather',
  'args': {'location': 'San Francisco, CA'},
  'id': 'call_a8669dd5697f488a8c5760e1d37b4528',
  'type': 'tool_call'}]

In [138]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings()

text = "This is a test document."

In [139]:
query_result = embeddings.embed_query(text)
query_result[:5]

ValueError: Error raised by inference API HTTP code: 404, {"error":"model \"llama2\" not found, try pulling it first"}

In [ ]:
doc_result = embeddings.embed_documents([text])
doc_result[0][:5]

In [ ]:
embeddings = OllamaEmbeddings(model="llama3")
text = "This is a test document."
query_result = embeddings.embed_query(text)

In [ ]:
query_result[:5]

In [ ]:
doc_result = embeddings.embed_documents([text])

In [ ]:
doc_result[0][:5]